# Binary Classifier — Insect vs Background

Trains a binary insect vs background classifier on manually annotated crops from `annotate.py`.

## Two backbones compared
| | EfficientNet-B0 | InsectNet backbone |
|---|---|---|
| Input | 128×128 px | 224×224 px |
| Pretrained | ImageNet (1.2M images) | 6M insect species images |
| Trainable | All layers | Final fc only (frozen backbone) |
| Advantage | Fast, lightweight | Insect-specific features |

**Domain shift note:** InsectNet was trained on clean, centred insect photos. Arctic field crops are noisy and motion-blurred. Freeze the backbone first (`MODEL = 'insectnet'`). If validation recall is low, unfreeze the last block with a small lr (see Cell 3).

## Workflow
1. Run `ls_v4_1.ipynb` batch → crops
2. Run `annotate.py` → `labeled/insect/` and `labeled/background/`
3. Run this notebook → `models/binary_best.pth`

**Minimum data:** ~500 insect crops, ~1000 background crops (include tiles and context crops, not only tight crops).


In [ ]:
import json
import time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:
    from sklearn.metrics import classification_report, confusion_matrix
    HAS_SKLEARN = True
except ImportError:
    print('Install sklearn: pip install scikit-learn')
    HAS_SKLEARN = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')


## Configuration
Change these paths and settings before running.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
LABELED_DIR       = Path('Insects_images/labeled')   # output of annotate.py
MODEL_OUT_DIR     = Path('models')
INSECTNET_WEIGHTS = Path('InsectNet/model.pth')       # only needed for insectnet
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model choice ───────────────────────────────────────────────────────────
# 'efficientnet' → EfficientNet-B0, 128px, fast, all layers trainable
# 'insectnet'    → InsectNet backbone (RegNet-Y-32GF), 224px, frozen backbone
# 'both'         → train both and compare
MODEL = 'insectnet'

# ── Training settings ──────────────────────────────────────────────────────
EPOCHS   = 20
BATCH    = 32
LR       = 1e-3     # use 1e-5 if unfreezing backbone blocks
VAL_FRAC = 0.2
SEED     = 42

# ── Partial fine-tune (optional, use after initial frozen training) ─────────
# Set UNFREEZE_LAST_BLOCK = True to unfreeze the last backbone block + fc.
# Use a smaller LR (1e-5) to avoid destroying pretrained insect features.
# Only do this if frozen training recall is unsatisfactory.
UNFREEZE_LAST_BLOCK = False

# ── Verify labeled data ────────────────────────────────────────────────────
for cls in ['insect', 'background']:
    d = LABELED_DIR / cls
    n = len(list(d.glob('*.jpg'))) + len(list(d.glob('*.png'))) if d.exists() else 0
    status = '✓' if n >= (500 if cls == 'insect' else 1000) else f'← need more (target: {500 if cls=="insect" else 1000})'
    print(f'{cls:12}: {n:>5} images  {status}')


## Dataset and Splits

In [ ]:
CLASSES = ['background', 'insect']  # index 0=background, 1=insect
torch.manual_seed(SEED)
np.random.seed(SEED)


class CropDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label


def load_and_split(labeled_dir, val_frac, seed):
    all_samples = []
    for label_idx, cls in enumerate(CLASSES):
        d = labeled_dir / cls
        if not d.exists(): continue
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            for p in d.glob(ext):
                all_samples.append((p, label_idx))

    rng = np.random.default_rng(seed)
    by_class = {0: [], 1: []}
    for i, (_, lbl) in enumerate(all_samples):
        by_class[lbl].append(i)

    train_idx, val_idx = [], []
    for lbl, idxs in by_class.items():
        idxs = list(idxs)
        rng.shuffle(idxs)
        n_val = max(1, int(len(idxs) * val_frac))
        val_idx.extend(idxs[:n_val])
        train_idx.extend(idxs[n_val:])

    counts = {0: len(by_class[0]), 1: len(by_class[1])}
    print(f'background: {counts[0]}  |  insect: {counts[1]}')
    print(f'train: {len(train_idx)}  |  val: {len(val_idx)}')
    ratio = counts[0] / max(1, counts[1])
    print(f'imbalance ratio  background:insect = {ratio:.1f}:1')
    if counts[1] < 150:
        print('WARNING: fewer than 150 insect crops — annotate more before training')
    return all_samples, train_idx, val_idx, counts


def make_loaders(all_samples, train_idx, val_idx, counts, img_size, batch):
    train_tf = T.Compose([
        T.Resize((img_size, img_size)),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomRotation(30),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        # GaussianBlur simulates motion blur common in field crops
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    val_tf = T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    train_samples = [all_samples[i] for i in train_idx]
    val_samples   = [all_samples[i] for i in val_idx]

    train_labels = [s[1] for s in train_samples]
    cls_w = 1.0 / np.maximum(np.bincount(train_labels, minlength=2), 1)
    sample_w = [cls_w[l] for l in train_labels]
    sampler = WeightedRandomSampler(sample_w, len(sample_w))

    train_loader = DataLoader(CropDataset(train_samples, train_tf),
                              batch_size=batch, sampler=sampler, num_workers=0)
    val_loader   = DataLoader(CropDataset(val_samples, val_tf),
                              batch_size=batch, shuffle=False, num_workers=0)
    return train_loader, val_loader


all_samples, train_idx, val_idx, counts = load_and_split(LABELED_DIR, VAL_FRAC, SEED)


## Model Builders

In [ ]:
def build_efficientnet():
    """EfficientNet-B0 pretrained on ImageNet. All layers trainable."""
    print('Building EfficientNet-B0 (ImageNet, 128px, all layers)')
    model = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(1280, 2)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Trainable params: {n:,}')
    return model, 128


def build_insectnet(weights_path, unfreeze_last_block=False):
    """
    InsectNet backbone (RegNet-Y-32GF) with frozen backbone.

    Default: freeze backbone, train only final fc.
    Good for small datasets — avoids overfitting.

    Set unfreeze_last_block=True to also unfreeze the last backbone block
    for partial fine-tuning when domain shift is large (use lr=1e-5).
    """
    print(f'Building InsectNet binary (RegNet-Y-32GF, 224px)')
    if not weights_path.exists():
        raise FileNotFoundError(f'InsectNet weights not found: {weights_path}')
    print(f'Loading weights: {weights_path}')

    model = torchvision.models.regnet_y_32gf()
    model.fc = nn.Linear(3712, 2526)
    state = torch.load(weights_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state['model'] if 'model' in state else state, strict=True)
    print('InsectNet weights loaded ✓')

    # Replace classifier head for binary task
    model.fc = nn.Linear(3712, 2)
    nn.init.xavier_uniform_(model.fc.weight)
    nn.init.zeros_(model.fc.bias)

    # Freeze all backbone weights
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith('fc.')

    # Optionally unfreeze last backbone block for partial fine-tuning
    if unfreeze_last_block:
        print('Unfreezing last backbone block (trunk_output.block4) + fc')
        for name, p in model.named_parameters():
            if 'trunk_output.block4' in name or name.startswith('fc.'):
                p.requires_grad = True

    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {n_train:,} / {n_total:,} params '
          f'({"frozen backbone" if not unfreeze_last_block else "last block + fc unfrozen"})')
    return model, 224


## Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum = correct = total = tp = fn = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        preds = out.argmax(1)
        loss_sum += loss.item() * labels.size(0)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)
        m = (labels == 1)
        tp += (preds[m] == 1).sum().item()
        fn += (preds[m] == 0).sum().item()
    return loss_sum/total, correct/total, tp/max(1, tp+fn)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    loss_sum = correct = total = tp = fp = fn = 0
    all_p, all_l = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out   = model(imgs)
        preds = out.argmax(1)
        loss_sum += criterion(out, labels).item() * labels.size(0)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)
        all_p.extend(preds.cpu().tolist())
        all_l.extend(labels.cpu().tolist())
        tp += ((preds==1)&(labels==1)).sum().item()
        fp += ((preds==1)&(labels==0)).sum().item()
        fn += ((preds==0)&(labels==1)).sum().item()
    prec = tp / max(1, tp+fp)
    rec  = tp / max(1, tp+fn)
    f1   = 2*prec*rec / max(1e-8, prec+rec)
    return {'loss': loss_sum/total, 'acc': correct/total,
            'precision': prec, 'recall': rec, 'f1': f1,
            'preds': all_p, 'labels': all_l}


def run_training(model, model_name, img_size, counts):
    """Full training loop — saves best model, plots curves, prints report."""
    train_loader, val_loader = make_loaders(
        all_samples, train_idx, val_idx, counts, img_size, BATCH)

    model = model.to(DEVICE)

    # Weighted loss — insect class gets higher weight
    w = torch.tensor([1/max(1,counts[0]), 1/max(1,counts[1])],
                     dtype=torch.float, device=DEVICE)
    w = w / w.sum()
    criterion = nn.CrossEntropyLoss(weight=w)

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_f1   = 0.0
    ckpt_path = MODEL_OUT_DIR / f'{model_name}_binary_best.pth'
    history   = {k: [] for k in ['tr_loss','val_loss','tr_rec','val_rec',
                                  'val_prec','val_f1','val_acc']}

    print(f'\n{"="*65}')
    print(f'{model_name}  |  img={img_size}px  |  epochs={EPOCHS}  |  lr={LR}')
    print(f'{"="*65}')
    print(f'{"Ep":>3}  {"TrLoss":>7}  {"VaLoss":>7}  '
          f'{"Prec":>6}  {"Recall":>7}  {"F1":>6}  {"Acc":>6}')

    t0 = time.time()
    for ep in range(1, EPOCHS+1):
        tr_loss, tr_acc, tr_rec = train_epoch(model, train_loader,
                                              optimizer, criterion, DEVICE)
        v = eval_epoch(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history['tr_loss'].append(tr_loss)
        history['val_loss'].append(v['loss'])
        history['tr_rec'].append(tr_rec)
        history['val_rec'].append(v['recall'])
        history['val_prec'].append(v['precision'])
        history['val_f1'].append(v['f1'])
        history['val_acc'].append(v['acc'])

        star = ''
        if v['f1'] > best_f1:
            best_f1 = v['f1']
            torch.save({'epoch': ep, 'model_name': model_name,
                        'img_size': img_size, 'state_dict': model.state_dict(),
                        'val_f1': v['f1'], 'val_recall': v['recall'],
                        'val_precision': v['precision']}, ckpt_path)
            star = ' *'

        print(f'{ep:>3}  {tr_loss:>7.4f}  {v["loss"]:>7.4f}  '
              f'{v["precision"]:>6.3f}  {v["recall"]:>7.3f}  '
              f'{v["f1"]:>6.3f}  {v["acc"]:>6.3f}{star}')

    print(f'\nDone in {(time.time()-t0)/60:.1f} min  |  '
          f'Best F1: {best_f1:.3f}  |  Saved: {ckpt_path}')

    # ── Curves ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'{model_name} Training', fontsize=13)
    axes[0].plot(history['tr_loss'], label='train')
    axes[0].plot(history['val_loss'], label='val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(history['tr_rec'],  label='train recall')
    axes[1].plot(history['val_rec'], label='val recall')
    axes[1].plot(history['val_prec'],label='val precision')
    axes[1].set_title('Recall / Precision (insect)'); axes[1].legend()
    axes[2].plot(history['val_f1'],  label='F1')
    axes[2].plot(history['val_acc'], label='Accuracy')
    axes[2].set_title('F1 / Accuracy'); axes[2].legend()
    plt.tight_layout()
    curve_path = MODEL_OUT_DIR / f'{model_name}_curves.png'
    plt.savefig(curve_path, dpi=100); plt.close()
    print(f'Curves saved: {curve_path}')

    # ── Final report ──────────────────────────────────────────────────────────
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['state_dict'])
    final = eval_epoch(model, val_loader, criterion, DEVICE)

    print(f'\n--- Final report (best checkpoint epoch {ckpt["epoch"]}) ---')
    if HAS_SKLEARN:
        print(classification_report(final['labels'], final['preds'],
                                    target_names=CLASSES, digits=3))
        cm = confusion_matrix(final['labels'], final['preds'])
        print(f'Confusion matrix:')
        print(f'                  pred:bg  pred:insect')
        print(f'  true:background  {cm[0,0]:>5}     {cm[0,1]:>5}   (false positives)')
        print(f'  true:insect      {cm[1,0]:>5}     {cm[1,1]:>5}   (false negatives = missed insects)')

    results = {'model': model_name, 'img_size': img_size,
               'best_epoch': ckpt['epoch'], 'val_f1': final['f1'],
               'val_recall': final['recall'], 'val_precision': final['precision'],
               'val_acc': final['acc']}
    (MODEL_OUT_DIR / f'{model_name}_results.json').write_text(
        json.dumps(results, indent=2))
    return results


## Run Training
Set `MODEL` in the Config cell to `'efficientnet'`, `'insectnet'`, or `'both'`.

In [ ]:
results = {}

if MODEL in ('efficientnet', 'both'):
    model_eff, img_eff = build_efficientnet()
    results['efficientnet'] = run_training(model_eff, 'efficientnet', img_eff, counts)

if MODEL in ('insectnet', 'both'):
    if not INSECTNET_WEIGHTS.exists():
        print(f'ERROR: InsectNet weights not found at {INSECTNET_WEIGHTS}')
        print('Set INSECTNET_WEIGHTS to the correct path.')
    else:
        model_ins, img_ins = build_insectnet(INSECTNET_WEIGHTS)
        results['insectnet'] = run_training(model_ins, 'insectnet', img_ins, counts)

if MODEL == 'both' and len(results) == 2:
    print('\n' + '='*65)
    print('COMPARISON SUMMARY')
    print('='*65)
    print(f'{"Model":20} {"F1":>6}  {"Recall":>7}  {"Precision":>10}  {"Acc":>6}')
    for name, r in results.items():
        print(f'{name:20} {r["val_f1"]:>6.3f}  {r["val_recall"]:>7.3f}  '
              f'{r["val_precision"]:>10.3f}  {r["val_acc"]:>6.3f}')
    print('\nPrioritise Recall — missing a real insect is worse than a false positive.')


## Test: Run Classifier on a Single Crop
Quick sanity check — load a crop and see what the classifier predicts.

In [ ]:
def load_classifier(ckpt_path):
    """Load a saved binary classifier for inference."""
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model_name = ckpt['model_name']
    img_size   = ckpt['img_size']

    if model_name == 'efficientnet':
        model = torchvision.models.efficientnet_b0(weights=None)
        model.classifier[-1] = nn.Linear(1280, 2)
    else:
        model = torchvision.models.regnet_y_32gf()
        model.fc = nn.Linear(3712, 2)

    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    print(f'Loaded {model_name} from {ckpt_path}')
    print(f'  val F1={ckpt["val_f1"]:.3f}  recall={ckpt["val_recall"]:.3f}')
    return model, img_size


def predict_crop(model, img_size, crop_path):
    """Predict insect or background for a single crop image."""
    tf = T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img = Image.open(crop_path).convert('RGB')
    x   = tf(img).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    bg_prob  = probs[0].item()
    ins_prob = probs[1].item()
    pred     = 'insect' if ins_prob > bg_prob else 'background'
    print(f'Prediction: {pred}  (insect={ins_prob:.2%}  background={bg_prob:.2%})')
    return pred, ins_prob


# ── Example usage ─────────────────────────────────────────────────────────────
ckpt_path = MODEL_OUT_DIR / 'efficientnet_binary_best.pth'
if ckpt_path.exists():
    clf, img_sz = load_classifier(ckpt_path)

    # Replace with any crop path from your results folder
    test_crop = Path('Insects_images/labeled/insect').glob('*.jpg')
    test_crop = next(test_crop, None)
    if test_crop:
        print(f'Testing on: {test_crop.name}')
        predict_crop(clf, img_sz, test_crop)
    else:
        print('No crops found in labeled/insect/ — run annotate.py first')
else:
    print(f'No checkpoint found at {ckpt_path} — run the training cell first')
